# De vuistregeltools testen

Elke maatregel is een mutatie van de polder-toestand. Vooruit rekenen (`given_area`,
`given_discharge`) en achteruit rekenen (`given_h`) lopen over dezelfde `simulate`, dus het
rondje sluit binnen het haalbare bereik.

Let op de eenheden: alle cellen hieronder gebruiken `discharge_capacity_m3_per_sec=5`.
Eerdere versies van deze notebook gebruikten bij bergen `m3_per_min=5`, een factor 60
kleiner, waardoor die cellen onvergelijkbare uitkomsten gaven.

In [1]:
import logging
import textwrap

from talk_with_the_engineer.routers import get_runtime_ctx

ctx = get_runtime_ctx()
logging.getLogger("AI_agent").setLevel(logging.WARNING)


def show(output):
    print(textwrap.fill(output.description, width=88))
    print()
    print(f"  status              {output.status.value}")
    if output.limit_reason.value != "none":
        print(f"  begrenzing          {output.limit_reason.value}")
    print(f"  zonder maatregel    {output.dh_reference_m * 100:7.1f} cm")
    print(f"  met maatregel       {output.dh_with_measure_m * 100:7.1f} cm")
    print(f"  verlaging           {output.dh_reduction_m * 100:7.1f} cm")
    print(
        f"  haalbaar bereik     {output.feasible_dh_min_m * 100:7.1f} .. "
        f"{output.feasible_dh_max_m * 100:.1f} cm"
    )
    print(
        f"  open water          {output.state_before.pct_open_water:7.2f} -> "
        f"{output.state_after.pct_open_water:.2f} %"
    )
    print(
        f"  gemaalcapaciteit    {output.state_before.pump_capacity_m3_per_sec:7.2f} -> "
        f"{output.state_after.pump_capacity_m3_per_sec:.2f} m3/s"
    )
    print(
        f"  retentie            {output.state_before.retention_fraction * 100:7.1f} -> "
        f"{output.state_after.retention_fraction * 100:.1f} % van het land"
    )

## De referentiecasus uit Ton's rekenblad

`scripts/scripts_Ton/TttE_v03.xlsx` rekent met deze bui en polder. De waterstandstijging
zonder maatregel is 1,70233 m; de tools reproduceren alle drie de maatregelcurves tot op
machineprecisie. Let op dat dit **niet** de defaults zijn: die staan op `p1c` en
`days_to_discharge=3`.

In [2]:
from talk_with_the_engineer.models import StoreInputGivenH
from talk_with_the_engineer.models.modelset_rules_of_thumb.state import PolderState
from talk_with_the_engineer.tools.toolset_rules_of_thumb.simulate import simulate

REFERENCE_STORM = dict(
    return_period=1000,
    time_of_year="winter",
    length_event_hrs=96.0,
    name_rainfall_pattern="p1a",
    days_to_discharge=4,
    rainfall_regime="R",
)
REFERENCE_POLDER = dict(
    area_ha=4000,
    percentage_open_water=3.5,
    discharge_capacity_m3_per_sec=5,
)
REFERENCE = {**REFERENCE_STORM, **REFERENCE_POLDER}

storm = StoreInputGivenH(delta_h_available=1.0, **REFERENCE)
state = PolderState.from_water_system_input(storm)
print(
    f"polder            {state.area_ha:.0f} ha, {state.area_open_water_ha:.0f} ha open water, "
    f"{state.area_land_ha:.0f} ha land"
)
print(
    f"waterstandstijging {simulate(state, storm).dh_max_m:.5f} m   (rekenblad 1.70233)"
)

polder            4000 ha, 140 ha open water, 3860 ha land
waterstandstijging 1.70233 m   (rekenblad 1.70233)


### Store (bergen) — land wordt open water

In [3]:
from talk_with_the_engineer.models import StoreInputGivenH
from talk_with_the_engineer.tools import calculate_store_given_h

# achteruit: welke maatvoering hoort bij een doelwaterstand?
show(
    await calculate_store_given_h(
        ctx, StoreInputGivenH(delta_h_available=1.0, **REFERENCE)
    )
)

Om de waterstandstijging bij een bui van eens per 1000 jaar op 100 cm te houden, is met
de maatregel 'bergen' 98 ha extra open water nodig. Zonder maatregel is de stijging 170
cm, dus dat is een verlaging van 70 cm.

  status              exact
  zonder maatregel      170.2 cm
  met maatregel         100.0 cm
  verlaging              70.2 cm
  haalbaar bereik         6.0 .. 170.2 cm
  open water             3.50 -> 5.96 %
  gemaalcapaciteit       5.00 -> 5.00 m3/s
  retentie                0.0 -> 0.0 % van het land


In [4]:
from talk_with_the_engineer.models import StoreInputGivenArea
from talk_with_the_engineer.tools import calculate_store_given_area

# vooruit: het antwoord van hierboven teruggevoerd - het rondje sluit
show(
    await calculate_store_given_area(
        ctx, StoreInputGivenArea(area_available_ha=98.3, **REFERENCE)
    )
)

Met 98 ha extra open water brengt de maatregel 'bergen' de waterstandstijging bij een
bui van eens per 1000 jaar op 100 cm. Zonder maatregel is de stijging 170 cm, dus dat is
een verlaging van 70 cm.

  status              exact
  zonder maatregel      170.2 cm
  met maatregel         100.0 cm
  verlaging              70.2 cm
  haalbaar bereik         6.0 .. 170.2 cm
  open water             3.50 -> 5.96 %
  gemaalcapaciteit       5.00 -> 5.00 m3/s
  retentie                0.0 -> 0.0 % van het land


### Discharge (afvoeren) — meer gemaalcapaciteit

In [5]:
from talk_with_the_engineer.models import (
    DischargeInputGivenDischarge,
    DischargeInputGivenH,
)
from talk_with_the_engineer.tools import (
    calculate_discharge_given_discharge,
    calculate_discharge_given_h,
)

show(
    await calculate_discharge_given_h(
        ctx, DischargeInputGivenH(delta_h_available=1.0, **REFERENCE)
    )
)

Om de waterstandstijging bij een bui van eens per 1000 jaar op 100 cm te houden, is met
de maatregel 'afvoeren' 2.49 m3/s extra gemaalcapaciteit nodig. Zonder maatregel is de
stijging 170 cm, dus dat is een verlaging van 70 cm.

  status              exact
  zonder maatregel      170.2 cm
  met maatregel         100.0 cm
  verlaging              70.2 cm
  haalbaar bereik         0.0 .. 170.2 cm
  open water             3.50 -> 3.50 %
  gemaalcapaciteit       5.00 -> 7.49 m3/s
  retentie                0.0 -> 0.0 % van het land


In [6]:
show(
    await calculate_discharge_given_discharge(
        ctx,
        DischargeInputGivenDischarge(
            delta_discharge_capacity_m3_per_sec=2.49, **REFERENCE
        ),
    )
)

Met 2.49 m3/s extra gemaalcapaciteit brengt de maatregel 'afvoeren' de
waterstandstijging bij een bui van eens per 1000 jaar op 100 cm. Zonder maatregel is de
stijging 170 cm, dus dat is een verlaging van 70 cm.

  status              exact
  zonder maatregel      170.2 cm
  met maatregel         100.1 cm
  verlaging              70.2 cm
  haalbaar bereik         0.0 .. 170.2 cm
  open water             3.50 -> 3.50 %
  gemaalcapaciteit       5.00 -> 7.49 m3/s
  retentie                0.0 -> 0.0 % van het land


### Retain (vasthouden) — de afvoer van een deel van het land vertragen

Deze maatregel heeft een **optimum**. Voorbij een zeker areaal valt de vertraagde afvoer
van het ene deel samen met de piek van het andere en wordt het weer slechter. Dat staat ook
zo in Ton's rekenblad: minimum 1,46183 m bij 70 %, en 1,51116 m bij 100 %.

In [7]:
from talk_with_the_engineer.models import RetentionInputGivenArea, RetentionInputGivenH
from talk_with_the_engineer.tools import (
    calculate_retention_given_area,
    calculate_retention_given_h,
)

# 30 % van het landoppervlak: 0,30 x 3860 = 1158 ha, zoals in het rekenblad
show(
    await calculate_retention_given_area(
        ctx, RetentionInputGivenArea(delta_area_ha=1158, **REFERENCE)
    )
)

Met 1158 ha land brengt de maatregel 'vasthouden' de waterstandstijging bij een bui van
eens per 1000 jaar op 159 cm. Zonder maatregel is de stijging 170 cm, dus dat is een
verlaging van 11 cm.

  status              exact
  zonder maatregel      170.2 cm
  met maatregel         158.7 cm
  verlaging              11.5 cm
  haalbaar bereik       146.0 .. 170.2 cm
  open water             3.50 -> 3.50 %
  gemaalcapaciteit       5.00 -> 5.00 m3/s
  retentie                0.0 -> 30.0 % van het land


In [8]:
# een doel onder het optimum: de tool geeft geen getal maar een uitleg
show(
    await calculate_retention_given_h(
        ctx, RetentionInputGivenH(delta_h_available=1.0, **REFERENCE)
    )
)

Een stijging van 100 cm is met alleen 'vasthouden' niet haalbaar: de maatregel bereikt
zijn optimum bij 146 cm, met 2657 ha land. Combineer met bergen of afvoeren om verder te
komen.

  status              infeasible
  begrenzing          measure_optimum
  zonder maatregel      170.2 cm
  met maatregel         146.0 cm
  verlaging              24.2 cm
  haalbaar bereik       146.0 .. 170.2 cm
  open water             3.50 -> 3.50 %
  gemaalcapaciteit       5.00 -> 5.00 m3/s
  retentie                0.0 -> 68.8 % van het land


## Buiten het haalbare bereik

Er zijn drie statussen. Voorheen werd er met `np.interp` geinterpoleerd, en die klemt stil
op de randen: een doel onder het haalbare leverde "0 ha nodig" op en een doel erboven
"100 %", zonder melding.

In [9]:
# doel boven de referentiestijging: geen maatregel nodig
show(
    await calculate_store_given_h(
        ctx, StoreInputGivenH(delta_h_available=2.5, **REFERENCE)
    )
)

De maatregel 'bergen' is niet nodig: bij een bui van eens per 1000 jaar stijgt de
waterstand 170 cm, en dat blijft al onder de beschikbare 250 cm.

  status              no_measure_needed
  zonder maatregel      170.2 cm
  met maatregel         170.2 cm
  verlaging               0.0 cm
  haalbaar bereik         6.0 .. 170.2 cm
  open water             3.50 -> 3.50 %
  gemaalcapaciteit       5.00 -> 5.00 m3/s
  retentie                0.0 -> 0.0 % van het land


In [10]:
# doel onder wat bergen hier kan bereiken
show(
    await calculate_store_given_h(
        ctx, StoreInputGivenH(delta_h_available=0.01, **REFERENCE)
    )
)

Een stijging van 1 cm is met alleen 'bergen' niet haalbaar: de fysieke grens van het
watersysteem is bereikt bij 6 cm, met 3860 ha extra open water. Combineer met afvoeren
of vasthouden om verder te komen.

  status              infeasible
  begrenzing          physical_upper_bound
  zonder maatregel      170.2 cm
  met maatregel           6.0 cm
  verlaging             164.3 cm
  haalbaar bereik         6.0 .. 170.2 cm
  open water             3.50 -> 100.00 %
  gemaalcapaciteit       5.00 -> 5.00 m3/s
  retentie                0.0 -> 0.0 % van het land


## De bui doet iets — hetzelfde doel bij vier terugkeertijden

In [11]:
print(f"{'T [jaar]':>9}{'neerslag':>11}{'zonder mtr':>13}{'nodig':>12}{'status':>20}")
for return_period in (10, 100, 250, 1000):
    output = await calculate_store_given_h(
        ctx,
        StoreInputGivenH(
            delta_h_available=0.5,
            **{**REFERENCE, "return_period": return_period},
        ),
    )
    print(
        f"{return_period:>9}{output.data['volume_rain_mm']:>9.1f} mm"
        f"{output.dh_reference_m * 100:>10.1f} cm"
        f"{output.data['delta_open_water_ha']:>9.0f} ha"
        f"{output.status.value:>20}"
    )

 T [jaar]   neerslag   zonder mtr       nodig              status
       10     73.0 mm      40.5 cm        0 ha   no_measure_needed
      100    100.4 mm      98.0 cm      134 ha               exact
      250    111.7 mm     126.3 cm      214 ha               exact
     1000    129.2 mm     170.2 cm      337 ha               exact
